# Practical 04: Out-of-Core Batch Ingestion & Aggregation via Chunking

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to ingest a large CSV file exceeding single-process RAM in batches using the chunksize parameter of pd.read_csv(). Calculate the cumulative record count, running sum of transaction amounts, and overall average transaction value across all chunks to prevent Out-Of-Memory (OOM) crashes.

In [1]:
import os
import shutil
import tempfile
import numpy as np
import pandas as pd

TOTAL_ROWS = 1_200_000
WRITE_BATCH = 300_000
CHUNK_SIZE = 150_000

rng = np.random.default_rng(11)
tmp_dir = tempfile.mkdtemp()
csv_path = os.path.join(tmp_dir, "large_transactions.csv")

written = 0
while written < TOTAL_ROWS:
    m = min(WRITE_BATCH, TOTAL_ROWS - written)
    batch = pd.DataFrame({
        "transaction_id": np.arange(written + 1, written + m + 1),
        "customer_id": rng.integers(1, 80000, m),
        "amount": np.round(rng.gamma(2.5, 50.0, m), 2),
        "city": rng.choice(["Chennai", "Kochi", "Pune", "Mumbai", "Hyderabad"], m),
    })
    batch.to_csv(csv_path, mode="a", header=(written == 0), index=False)
    written += m

print(f"File created: {os.path.getsize(csv_path) / 1024**2:.1f} MB, {TOTAL_ROWS:,} rows")

n_records = 0
sum_amount = 0.0

chunk_iter = pd.read_csv(csv_path, usecols=["transaction_id", "amount"], chunksize=CHUNK_SIZE)

print(f"{'Chunk':>5} | {'Rows':>8} | {'Cum. records':>12} | {'Running sum':>16} | {'Running avg':>11}")
print("-" * 66)
for i, chunk in enumerate(chunk_iter, start=1):
    n_records += len(chunk)
    sum_amount += chunk["amount"].sum()
    print(f"{i:>5} | {len(chunk):>8,} | {n_records:>12,} | {sum_amount:>16,.2f} | "
          f"{sum_amount / n_records:>11.4f}")

avg_value = sum_amount / n_records
print(f"Total records             : {n_records:,}")
print(f"Total transaction amount  : {sum_amount:,.2f}")
print(f"Average transaction value : {avg_value:.4f}")

full = pd.read_csv(csv_path, usecols=["amount"])["amount"]
print(f"Full-load sum : {full.sum():,.2f} | mean : {full.mean():.4f}")

shutil.rmtree(tmp_dir)
print("Temporary files removed.")

File created: 30.4 MB, 1,200,000 rows
Chunk |     Rows | Cum. records |      Running sum | Running avg
------------------------------------------------------------------
    1 |  150,000 |      150,000 |    18,735,656.53 |    124.9044
    2 |  150,000 |      300,000 |    37,536,065.80 |    125.1202
    3 |  150,000 |      450,000 |    56,316,914.57 |    125.1487
    4 |  150,000 |      600,000 |    75,066,030.39 |    125.1101
    5 |  150,000 |      750,000 |    93,861,372.41 |    125.1485
    6 |  150,000 |      900,000 |   112,626,445.99 |    125.1405
    7 |  150,000 |    1,050,000 |   131,389,903.93 |    125.1332
    8 |  150,000 |    1,200,000 |   150,124,661.39 |    125.1039
Total records             : 1,200,000
Total transaction amount  : 150,124,661.39
Average transaction value : 125.1039
Full-load sum : 150,124,661.39 | mean : 125.1039
Temporary files removed.